# Load Address Prediction Results

This notebook reads `results.csv` and plots runtime vs. iterations for each access-pattern and core-class combination.

In [1]:
import altair as alt
import duckdb

In [2]:
con = duckdb.connect()

df = con.sql("""
    SELECT
        iters AS iterations,
        pattern,
        core_kind,
        min,
        p25,
        median,
        p75,
        max,
        first,
        last,
        unit
    FROM read_csv_auto('results.csv')
    ORDER BY core_kind, pattern, iterations
""").df()

df.head()

,iterations,pattern,core_kind,min,p25,median,p75,max,first,last,unit
0,10,random,ecore,4391,4426,4472,4549,8727,8727,4437,cycles
1,20,random,ecore,4471,4503,4517,4537,4802,4665,4495,cycles
2,30,random,ecore,4539,4618,4642,4665,5044,4839,4629,cycles
3,40,random,ecore,4643,4723,4766,4810,22006,4984,4776,cycles
4,50,random,ecore,4770,4828,4883,4936,85536,5281,4878,cycles


In [3]:
unit = df["unit"].iloc[0]
y_title = f"Runtime ({unit})"

rs_df = df[df["pattern"].isin(["random", "strided"])]

chart = (
    alt.Chart(rs_df)
    .mark_line(point=True)
    .encode(
        x=alt.X("iterations:Q", title="Iterations"),
        y=alt.Y("median:Q", title=y_title),
        color=alt.Color("pattern:N", title="Pattern"),
        tooltip=[
            "pattern", "core_kind", "iterations",
            "min", "p25", "median", "p75", "max",
            "first", "last", "unit",
        ],
    )
    .properties(width=380, height=280)
    .facet(column=alt.Column("core_kind:N", title="Core Class", sort=["ecore", "pcore"]))
    .properties(title="Random vs. Strided")
    .interactive()
)

chart

alt.FacetChart(...)

In [4]:
sa_rv_df = df[df["pattern"].isin(["strided", "sa_rv"])]

sa_rv_chart = (
    alt.Chart(sa_rv_df)
    .mark_line(point=True)
    .encode(
        x=alt.X("iterations:Q", title="Iterations"),
        y=alt.Y("median:Q", title=y_title),
        color=alt.Color("pattern:N", title="Pattern"),
        tooltip=[
            "pattern", "core_kind", "iterations",
            "min", "p25", "median", "p75", "max",
            "first", "last", "unit",
        ],
    )
    .properties(width=380, height=280)
    .facet(column=alt.Column("core_kind:N", title="Core Class", sort=["ecore", "pcore"]))
    .properties(title="Strided vs. SA+RV — if they match, speedup is address (not value) prediction")
    .interactive()
)

sa_rv_chart

alt.FacetChart(...)